In [0]:
# Databricks notebook source
# %run ./00_config

from pyspark.sql import SparkSession

# ---- Configuration (single place to change) ----
CATALOG = "dbacademy"
SCHEMA  = "labuser13955035_1772876383"

# All data lives in one schema under the shared catalog
# Volume where CSV files were uploaded
VOL_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/h2files"

# ---- Create schema if not exists ----
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# ---- Create volume if not exists ----
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.h2files")

# ---- Verify files are present ----
print(f"\n📁 Files in {VOL_PATH}:")
files = dbutils.fs.ls(VOL_PATH)
for f in files:
    print(f"  {f.name:45s}  {f.size:>12,} bytes")

print(f"\n✅ Bootstrap complete: catalog={CATALOG}, schema={SCHEMA}")
print(f"   Volume path: {VOL_PATH}")


<!-- Legend -->
**Legend:**  
A = Project Folder  
A1, A2, A3 = Subfolders  
F = Data Files  
S = Scripts  
N = Notebooks  
UC = Unity Catalog  
V = Volumes  
R = Raw Tables  
S1x = Silver Tables  
G = Gold Tables  
O = Ops Tables  

flowchart TB
  %% -----------------------------
  %% PROJECT TREE
  %% -----------------------------
  A[hydrogen-14day-project/]
    subgraph Project_Folders
      A1[upload_bundle/] 
        F1[entsoe_nl_2020_prices.csv]
        F2[entsoe_nl_2021_prices.csv]
        F3[entsoe_nl_2020_load.csv]
        F4[entsoe_nl_2021_load.csv]
        F5[entsoe_nl_2020_generation.csv]
        F6[entsoe_nl_2021_generation.csv]
        F7[era5_nl_2020_hourly.csv]
        F8[era5_nl_2021_hourly.csv]
        F9[h2stations_nl.csv]
      A2[scripts/]
        S1[download_era5_nl_2020_2021.py]
        S2[prepare_era5_hourly_nl.py]
        S3[validate_era5_grib.py]
        S4[fetch_h2stations_data.py]
      A3[notebooks/]
        N0[00_config]
        N1[01_uc_bootstrap]
        N2[02_bronze_ingest]
        N3[03_silver_clean]
        N4[04_silver_integrate]
        N5[05_gold_marts]
        N9[99_ops_log]
    end
  A --> A1
  A --> A2
  A --> A3
  A1 --> F1
  A1 --> F2
  A1 --> F3
  A1 --> F4
  A1 --> F5
  A1 --> F6
  A1 --> F7
  A1 --> F8
  A1 --> F9
  A2 --> S1
  A2 --> S2
  A2 --> S3
  A2 --> S4
  A3 --> N0
  A3 --> N1
  A3 --> N2
  A3 --> N3
  A3 --> N4
  A3 --> N5
  A3 --> N9

  %% -----------------------------
  %% UNITY CATALOG + VOLUMES
  %% -----------------------------
  UC[h2 catalog]
    subgraph Volumes
      V1[/Volumes/h2/raw_entsoe/landing]
      V2[/Volumes/h2/raw_era5/landing]
      V3[/Volumes/h2/raw_h2stations/landing]
    end
  UC --> V1
  UC --> V2
  UC --> V3
  F1 --> V1
  F2 --> V1
  F3 --> V1
  F4 --> V1
  F5 --> V1
  F6 --> V1
  F7 --> V2
  F8 --> V2
  F9 --> V3

  %% -----------------------------
  %% RAW TABLES
  %% -----------------------------
  V1 --> R1[h2.raw_entsoe.prices_2020_raw]
  V1 --> R2[h2.raw_entsoe.prices_2021_raw]
  V1 --> R3[h2.raw_entsoe.load_2020_raw]
  V1 --> R4[h2.raw_entsoe.load_2021_raw]
  V1 --> R5[h2.raw_entsoe.generation_2020_raw]
  V1 --> R6[h2.raw_entsoe.generation_2021_raw]
  V2 --> R7[h2.raw_era5.weather_2020_raw]
  V2 --> R8[h2.raw_era5.weather_2021_raw]
  V3 --> R9[h2.raw_h2stations.stations_nl_raw]

  %% -----------------------------
  %% SILVER TABLES
  %% -----------------------------
  R1 --> S11[h2.silver.entsoe_prices_clean]
  R2 --> S11
  R3 --> S12[h2.silver.entsoe_load_clean]
  R4 --> S12
  R5 --> S13[h2.silver.entsoe_generation_hourly_by_type]
  R6 --> S13
  S13 --> S14[h2.silver.entsoe_generation_hourly_wide]
  R7 --> S15[h2.silver.weather_clean]
  R8 --> S15
  R9 --> S16[h2.silver.h2stations_clean]
  S11 --> S17[h2.silver.h2_training_base]
  S12 --> S17
  S14 --> S17
  S15 --> S17

  %% -----------------------------
  %% GOLD TABLES
  %% -----------------------------
  S14 --> G1[h2.gold.generation_dependency_hourly]
  S16 --> G2[h2.gold.h2_station_status_summary]
  S17 --> G3[h2.gold.market_weather_hourly]
  S14 --> G4[h2.gold.renewable_mix_hourly]
  S14 --> G5[h2.gold.fossil_dependency_hourly]
  G3 --> G6[h2.gold.daily_ops_kpis]
  G1 --> G6
  G2 --> G6
  G3 --> G7[h2.gold.model_scoring_base]
  G1 --> G7

  %% -----------------------------
  %% OPS
  %% -----------------------------
  N9 --> O1[h2.ops.pipeline_run_log]

In [0]:
displayHTML("""
<pre style="font-family: monospace; white-space: pre;">
Hydrogen Project Data Map (Unity Catalog)
h2
├── raw_entsoe
│   ├── prices_2020_raw
│   ├── prices_2021_raw
│   ├── load_2020_raw
│   ├── load_2021_raw
│   ├── generation_2020_raw
│   └── generation_2021_raw
│
├── raw_era5
│   ├── weather_2020_raw
│   └── weather_2021_raw
│
├── raw_h2stations
│   └── stations_nl_raw
│
├── silver
│   ├── entsoe_prices_clean
│   ├── entsoe_load_clean
│   ├── entsoe_generation_hourly_by_type
│   ├── entsoe_generation_hourly_wide
│   ├── weather_clean
│   ├── h2stations_clean
│   └── h2_training_base
│
├── gold
│   ├── generation_dependency_hourly
│   ├── h2_station_status_summary
│   ├── market_weather_hourly
│   ├── renewable_mix_hourly
│   ├── fossil_dependency_hourly
│   ├── daily_ops_kpis
│   └── model_scoring_base
│
└── ops
    └── pipeline_run_log
</pre>
""")

Project Objects — What Each One Is
- upload_bundle/*
  - Flat upload-ready source files (ENTSO-E, ERA5, H2 stations) for UC Volumes.
- /Volumes/h2/raw_entsoe/landing
  - Landing area for ENTSO-E raw CSV files.
- /Volumes/h2/raw_era5/landing
  - Landing area for ERA5 hourly weather files.
- /Volumes/h2/raw_h2stations/landing
  - Landing area for H2 station snapshot files.
Raw layer (h2.raw_*)
- h2.raw_entsoe.prices_2020_raw, prices_2021_raw
  - Raw day-ahead price records (as ingested, with lineage columns).
- h2.raw_entsoe.load_2020_raw, load_2021_raw
  - Raw load + day-ahead load forecast records.
- h2.raw_entsoe.generation_2020_raw, generation_2021_raw
  - Raw generation by production type (quarter-hour source granularity).
- h2.raw_era5.weather_2020_raw, weather_2021_raw
  - Raw hourly weather features (temperature, wind components, pressure).
- h2.raw_h2stations.stations_nl_raw
  - Raw station inventory/status snapshot for NL.
Silver layer (h2.silver)
- entsoe_prices_clean
  - Parsed/typed hourly price table (event_time_utc, zone, price_eur_mwh).
- entsoe_load_clean
  - Cleaned load table aggregated to hourly level.
- entsoe_generation_hourly_by_type
  - Hourly generation by production_type (retains source mix detail).
- entsoe_generation_hourly_wide
  - Pivoted generation table: one row per hour+zone, type columns flattened.
- weather_clean
  - Cleaned hourly weather with derived wind_speed_ms.
- h2stations_clean
  - Clean station table with valid coords and status IDs.
- h2_training_base
  - Integrated modeling base (prices + load + generation + weather + time features).
Gold layer (h2.gold)
- generation_dependency_hourly
  - Renewable vs non-renewable MW and share KPIs by hour.
- h2_station_status_summary
  - Station KPI summary (total/in operation/planned/old, active ratio).
- market_weather_hourly
  - Dashboard-ready operational hourly table.
- renewable_mix_hourly
  - Renewable component breakdown (wind/solar/hydro/biomass).
- fossil_dependency_hourly
  - Fossil/non-renewable dependency breakdown and shares.
- daily_ops_kpis
  - Daily aggregate KPI view for management reporting.
- model_scoring_base
  - Curated feature set for model training/scoring use.
Ops layer
- h2.ops.pipeline_run_log
  - Pipeline observability table (run timestamp, stage, status, row counts, run_id).
Notebook roles
- 00_config: shared constants/params
- 01_uc_bootstrap: create catalog/schemas/volumes
- 02_bronze_ingest: load landing files to raw tables
- 03_silver_clean: parse/cast/clean/pivot
- 04_silver_integrate: create h2_training_base
- 05_gold_marts: create dashboard/model gold tables
- 99_ops_log: write run metadata to ops log